## Validation Datasets

The goal is to compare station data to ERA5 data and buoy data to GHRSST data to see how much error we are getting in a sample of regions in our models

From there, we can use that to figure out a reasonable "perturbation" for our model, changing 2m Temperature and SST accordingly to account for likely measurement error and use that to run ensemble models.

In [159]:
#import what you need
import xarray as xr #to read grib
import pandas as pd #for station data
import numpy as np #for subtracting arrays
from datetime import datetime, timedelta #to handle the dates
import statistics #to get mean and sd

In [161]:
## read in atmospheric data

#ERA5
era5_sfcMar = xr.load_dataset("../../01Data/SFC/ERA5_WRF_SFC_201703.grib", engine="cfgrib")
era5_sfcApr = xr.load_dataset("../../01Data/SFC/ERA5_WRF_SFC_201704.grib", engine="cfgrib")

#era5_plMar = xr.load_dataset("../../01Data/PLEV/ERA5_WRF_PLEV_201703.grib", engine="cfgrib")
#era5_plpr = xr.load_dataset("../../01Data/PLEV/ERA5_WRF_PLEV_201704.grib", engine="cfgrib")

#Station data
anglStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/AngletonLakeJackson/GHCNh_USW00012976_2017.psv", sep="|")
beauStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Beaumont/GHCNh_USW00000313_2017.psv", sep="|")
conrStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Conroe/GHCNh_USW00053902_2017.psv", sep="|")
galvStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Galveston/GHCNh_USW00012923_2017.psv", sep="|")
housStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Houston/GHCNh_USW00012975_2017.psv", sep="|")

skipping variable: paramId==170 shortName='stl2'
Traceback (most recent call last):
  File "C:\Users\rpkamakura\AppData\Local\anaconda3\envs\wrf\Lib\site-packages\cfgrib\dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "C:\Users\rpkamakura\AppData\Local\anaconda3\envs\wrf\Lib\site-packages\cfgrib\dataset.py", line 641, in dict_merge
    raise DatasetBuildError(
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='depthBelowLandLayer' value=Variable(dimensions=(), data=0.0) new_value=Variable(dimensions=(), data=7.0)
skipping variable: paramId==183 shortName='stl3'
Traceback (most recent call last):
  File "C:\Users\rpkamakura\AppData\Local\anaconda3\envs\wrf\Lib\site-packages\cfgrib\dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "C:\Users\rpkamakura\AppData\Local\anaconda3\envs\wrf\Lib\site-packages\cfgrib\dataset.py", line 641, in dict_merge
    raise DatasetBuildE

In [52]:
## Get useful station information for later work

#Station locations
lats = [anglStation["LATITUDE"][0], beauStation["LATITUDE"][0], conrStation["LATITUDE"][0], 
        galvStation["LATITUDE"][0], housStation["LATITUDE"][0]]
lons = [anglStation["LONGITUDE"][0], beauStation["LONGITUDE"][0], conrStation["LONGITUDE"][0], 
        galvStation["LONGITUDE"][0], housStation["LONGITUDE"][0]]

## Prep some other variables for comparisons
FocalTimePts = [datetime.fromisoformat("2017-03-23T00:00:00"), datetime.fromisoformat("2017-04-01T00:00:00"), 
                datetime.fromisoformat("2017-04-03T00:00:00")]

In [79]:
## Comparisons for Angleton Lake Jackson
lonTol = 0.13 #unlikely to get an exact match
latTol = 0.13 #unlikely to get an exact match

# Find subset within the tolerance range
anglSubsetMar = era5_sfcMar.t2m.sel(
    latitude=slice(lats[0] + latTol, lats[0] - latTol),
    longitude=slice(lons[0] - lonTol, lons[0] + lonTol),
    time=slice(FocalTimePts[0], FocalTimePts[1])
)

anglSubsetApr = era5_sfcApr.t2m.sel(
    latitude=slice(lats[0] + latTol, lats[0] - latTol),
    longitude=slice(lons[0] - lonTol, lons[0] + lonTol),
    time=slice(FocalTimePts[1], FocalTimePts[2])
)

In [162]:
#find corresponding temperature values within some tolerance of the era5 times

# Store average temperatures - March
mar_avg_temps = []
tolerance= timedelta(minutes=30)
anglStation = anglStation.to_dict('records')

#loop through it
for t in anglSubsetMar.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in anglStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and entry['temperature_Quality_Code'] == 5
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    mar_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
marDiffs = np.array([anglSubsetMar.values - 272.15]).flatten() - mar_avg_temps

# Store average temperatures - April
apr_avg_temps = []
tolerance= timedelta(minutes=30)

#loop through it
for t in anglSubsetApr.time.values:

    focalTime = pd.to_datetime(pd.Timestamp(t))
    # Subset data within the time tolerance
    subset = [
        entry['temperature'] for entry in anglStation
        if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and entry['temperature_Quality_Code'] == 5
    ]

    # Compute the average temperature if any entries matched
    if subset:
        avg = sum(subset) / len(subset)
    else:
        avg = float('nan')
        
    apr_avg_temps.append(avg)
    
#get the difference between these time points and the closest station values
aprDiffs = np.array([anglSubsetApr.values - 272.15]).flatten() - apr_avg_temps

TypeError: can't convert type 'ndarray' to numerator/denominator

In [168]:
#get some summary statistics
angl_avgDiff = statistics.mean([*marDiffs.tolist(), *aprDiffs.tolist()])
angl_sdDiff = statistics.stdev([*marDiffs.tolist(), *aprDiffs.tolist()])